In [1]:
import os
import sys
project_root = os.path.dirname(os.path.abspath('.'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
__package__ = 'rampa'

from rampa.query.duckdb_tools import DataManager, ArcGISQuery

In [2]:
import osmnx as ox
import pandas as pd
import geopandas as gpd
from rampa.routing.route import Network
import contextily as ctx

# Layers

When a database connection is not provided, DataManager behaves like Data_Collection

In [ ]:
dmanager = DataManager(json_file=project_root +'/Users/mherrero/Documents/madrid_open_data/rampa/data/urls.json', populate=True)
dmanager.download_data()

But if we provide a database, we can populate it

In [4]:
dmanager_db = DataManager(json_file=project_root + '/rampa/data/urls.json', 
                          db_connection=project_root + '/rampa/data/prueba.db', 
                          populate=True)

INFO:rampa.query.arcgis_tools:Loading URL dictionary from JSON file: /Users/mherrero/Documents/madrid_open_data/rampa/data/urls.json
INFO:rampa.query.duckdb_tools:Creating collection of layers from URL dictionary.
  0%|          | 0/87 [00:00<?, ?it/s]INFO:rampa.query.arcgis_tools:Creating FeatureLayer from URL https://sigma.madrid.es/hosted/rest/services/GEOPORTAL/AREAS_ACTIVIDADES_PARA_MAYORES/MapServer/0
INFO:rampa.query.duckdb_tools:Stored layer AREAS_ACTIVIDADES_PARA_MAYORES in database with 305 records.
  1%|          | 1/87 [00:12<18:06, 12.63s/it]INFO:rampa.query.arcgis_tools:Creating FeatureLayer from URL https://sigma.madrid.es/hosted/rest/services/GEOPORTAL/AREAS_ACTIVIDADES_PARA_MAYORES/MapServer/1
INFO:rampa.query.duckdb_tools:Stored layer EJERCICIOS_PARA_MAYORES in database with 200 records.
  2%|▏         | 2/87 [00:38<28:37, 20.20s/it]INFO:rampa.query.arcgis_tools:Creating FeatureLayer from URL https://sigma.madrid.es/hosted/rest/services/GEOPORTAL/ASEOS_PUBLICOS/MapSer

Alternatively, if the database already exist, we can just load the layers onto the manager

In [3]:
dmanager_read = DataManager(json_file=project_root + '/rampa/data/urls.json', 
                          db_connection=project_root + '/rampa/data/prueba.db', 
                          populate=False) 

INFO:rampa.query.arcgis_tools:Loading URL dictionary from JSON file: /Users/mherrero/Documents/madrid_open_data/rampa/data/urls.json


By default, the data is kept in the database and brought only to memory when queried

In [4]:
print(len(dmanager_read.data))
layer = dmanager_read.get_layer_data('AREAS_ACTIVIDADES_PARA_MAYORES')
print(len(dmanager_read.data))

0


INFO:rampa.query.duckdb_tools:Loaded layer AREAS_ACTIVIDADES_PARA_MAYORES from database: 305 records


1


We can load all layers in memory by

In [5]:
dmanager_read.load_all_layers()
print(len(dmanager_read.data))

  0%|          | 0/87 [00:00<?, ?it/s]INFO:rampa.query.duckdb_tools:Loaded layer EJERCICIOS_PARA_MAYORES from database: 200 records
INFO:rampa.query.duckdb_tools:Loaded layer ASEOS_PUBLICOS from database: 118 records
INFO:rampa.query.duckdb_tools:Loaded layer CUIDAR_A_QUIENES_NOS_CUIDAN from database: 21 records
INFO:rampa.query.duckdb_tools:Loaded layer JUNTOS_CONTRA_LA_SOLEDAD from database: 20 records
INFO:rampa.query.duckdb_tools:Loaded layer NOMBRES_DE_BARRIOS from database: 131 records
INFO:rampa.query.duckdb_tools:Loaded layer SECCIONES_CENSALES from database: 2462 records
INFO:rampa.query.duckdb_tools:Loaded layer DENSIDAD_POBLACION from database: 2450 records
INFO:rampa.query.duckdb_tools:Loaded layer EDAD_PROMEDIO from database: 2450 records
INFO:rampa.query.duckdb_tools:Loaded layer PROPORCION_JUVENTUD from database: 2450 records
INFO:rampa.query.duckdb_tools:Loaded layer PROPORCION_ENVEJECIMIENTO from database: 2450 records
INFO:rampa.query.duckdb_tools:Loaded layer PROPORC

87


# Routing

We can start by downloading the graph defined by a boundary. The optional argument db allows us to store the graph using duckdb.

In [3]:
lay = ArcGISQuery('https://sigma.madrid.es/hosted/rest/services/CARTOGRAFIA/ANCHO_MEDIO_ACERA/MapServer/0')
lay.create_layer()
aceras = lay.query(where="1=1")
madrid = ox.geocoder.geocode_to_gdf('R5326784', by_osmid=True)
net = Network(madrid, db=project_root + '/rampa/data/grafo.db', db_alt=project_root + '/rampa/data/grafo_alt.db', aceras=aceras, row='Ancho_medio', store=True)

INFO:rampa.query.arcgis_tools:Creating FeatureLayer from URL https://sigma.madrid.es/hosted/rest/services/CARTOGRAFIA/ANCHO_MEDIO_ACERA/MapServer/0


Generating contraction hierarchies with 1 threads.
Setting CH node vector of size 165387
Setting CH edge vector of size 495988
Range graph removed 501892 edges of 991976
. 10% . 20% . 30% . 40% . 50% . 60% . 70% . 80% . 90% . 100%


/Users/mherrero/Documents/madrid_open_data/rampa/routing/route.py:217: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.edges_alt[self.row] = self.edges_alt[self.row].fillna(0)


Generating contraction hierarchies with 1 threads.
Setting CH node vector of size 165387
Setting CH edge vector of size 495988
Range graph removed 501892 edges of 991976
. 10% . 20% . 30% . 40% . 50% . 60% . 70% . 80% . 90% . 100%
Storing network in DuckDB
Storing network in DuckDB


After it's saved into the DB, we can simply load it from there, without need to download it again

In [3]:
net_db = Network(db=project_root + '/rampa/data/grafo.db', db_alt=project_root + '/rampa/data/grafo_alt.db')

Generating contraction hierarchies with 1 threads.
Setting CH node vector of size 165387
Setting CH edge vector of size 495988
Range graph removed 501892 edges of 991976
. 10% . 20% . 30% . 40% . 50% . 60% . 70% . 80% . 90% . 100%
Generating contraction hierarchies with 1 threads.
Setting CH node vector of size 165387
Setting CH edge vector of size 495988
Range graph removed 501892 edges of 991976
. 10% . 20% . 30% . 40% . 50% . 60% . 70% . 80% . 90% . 100%


Nodes and edges are stored in their corresponding dataframes

In [4]:
net_db.nodes

,x,y
index,,
171946,-3.684443,40.421247
171951,-3.688989,40.417360
171952,-3.688980,40.414991
171953,-3.688901,40.412775
20952893,-3.597241,40.430465
...,...,...
13069374949,-3.759205,40.401607
13069761885,-3.688659,40.477519
13069761886,-3.688402,40.477475


In [5]:
net_db.edges

,from,to,distance
0,171946,11218044065,3.817747
1,171946,1209330272,33.546989
2,171946,3280496563,28.937732
3,171951,26486638,125.173331
4,171951,1209331009,18.732956
...,...,...,...
495983,13069761888,13069761885,18.114956
495984,13069761888,13069761889,22.058976
495985,13069761889,5676730807,70.251765
495986,13069761889,13069761888,22.058976


In [6]:
net_db.edges_alt

,from,to,distance,Ancho_medio,accesibility,alt_distance
0,171946,11218044065,3.817747,0.000000,0,3.817747
1,171946,1209330272,33.546989,4.245921,1,3.354699
2,171946,3280496563,28.937732,0.000000,0,28.937732
3,171951,26486638,125.173331,0.000000,0,125.173331
4,171951,1209331009,18.732956,0.000000,0,18.732956
...,...,...,...,...,...,...
495983,13069761888,13069761885,18.114956,0.000000,0,18.114956
495984,13069761888,13069761889,22.058976,0.000000,0,22.058976
495985,13069761889,5676730807,70.251765,0.000000,0,70.251765
495986,13069761889,13069761888,22.058976,0.000000,0,22.058976


The routing engine works by invoking route or route_gdf. The first returns a list of nodes, the second a gdf.

In [2]:
coord1 = (-3.703790, 40.416775)  # Example coordinates for Madrid in (longitude, latitude) format.
coord2 = (-3.72, 40.416775)

In [3]:
ruta = net_db.route_gdf(coord1, coord2, alternate=False)
ruta_alt = net_db.routÇe_gdf(coord1, coord2, alternate=True)

NameError: name 'net_db' is not defined

In [4]:
ax = ruta.plot(
    label=f"Rutas normal {ruta['distance'].sum():.0f} m",
    figsize=(10, 10)
)
ruta_alt.plot(
    ax=ax,
    color='red',
    label=f"Ruta accesible {ruta_alt['distance'].sum():.0f} m"
)

# Add legend explicitly
ax.legend()
ax.set_axis_off()

ctx.add_basemap(ax, crs=ruta.crs)


NameError: name 'ruta' is not defined

In [1]:
ruta

NameError: name 'ruta' is not defined